In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="1GhXHYlAY5Dqh7oZqpEF")
project = rf.workspace("jaa-ifc6q").project("ambulance-fire-police")
version = project.version(5)
dataset = version.download("yolov8")


In [ ]:
import os
import shutil
import random

# =========================
# Settings
# =========================

DATASET_DIR = "Ambulance-Fire-Police-5"

TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

SEED = 42

# =========================
# Paths
# =========================

images_dir = os.path.join(DATASET_DIR, "train", "images")
labels_dir = os.path.join(DATASET_DIR, "train", "labels")

splits = {
    "train": TRAIN_RATIO,
    "val": VAL_RATIO,
    "test": TEST_RATIO
}

# =========================
# Check ratios
# =========================

if TRAIN_RATIO + VAL_RATIO + TEST_RATIO != 1:
    raise ValueError("Train + Val + Test ratios must equal 1")

# =========================
# Get images
# =========================

image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

images = [
    f for f in os.listdir(images_dir)
    if f.lower().endswith(image_extensions)
]

print(f"Total images: {len(images)}")

# Shuffle
random.seed(SEED)
random.shuffle(images)

# =========================
# Calculate split sizes
# =========================

total = len(images)

train_count = int(total * TRAIN_RATIO)
val_count = int(total * VAL_RATIO)

train_images = images[:train_count]
val_images = images[train_count:train_count + val_count]
test_images = images[train_count + val_count:]

print(f"Train: {len(train_images)}")
print(f"Val:   {len(val_images)}")
print(f"Test:  {len(test_images)}")

# =========================
# Create folders
# =========================

for split in splits:

    os.makedirs(
        os.path.join(DATASET_DIR, split, "images"),
        exist_ok=True
    )

    os.makedirs(
        os.path.join(DATASET_DIR, split, "labels"),
        exist_ok=True
    )

# =========================
# Function to move image + label
# =========================

def move_data(image_list, split):

    for image_name in image_list:

        # Image source
        image_src = os.path.join(
            images_dir,
            image_name
        )

        # Label name
        label_name = os.path.splitext(image_name)[0] + ".txt"

        label_src = os.path.join(
            labels_dir,
            label_name
        )

        # Destination
        image_dst = os.path.join(
            DATASET_DIR,
            split,
            "images",
            image_name
        )

        label_dst = os.path.join(
            DATASET_DIR,
            split,
            "labels",
            label_name
        )

        # Move image
        shutil.move(image_src, image_dst)

        # Move label if exists
        if os.path.exists(label_src):

            shutil.move(label_src, label_dst)

        else:

            print(f"WARNING: Label not found for {image_name}")


# =========================
# Perform split
# =========================

move_data(train_images, "train")
move_data(val_images, "val")
move_data(test_images, "test")

print("\nDataset split completed successfully!")

In [ ]:
import os

dataset_path = "Ambulance-Fire-Police-5"

folders = ["train", "val", "test"]

for folder in folders:
    images_path = os.path.join(dataset_path, folder, "images")
    labels_path = os.path.join(dataset_path, folder, "labels")

    num_images = len(os.listdir(images_path))

    num_labels = len(os.listdir(labels_path))

    print(f"{folder}")
    print(f"   Images : {num_images}")
    print(f"   Labels : {num_labels}")
    print("-" * 30)

In [ ]:
import os
import random
import cv2
import matplotlib.pyplot as plt

# Dataset path
dataset_path = "Ambulance-Fire-Police-5"

images_path = os.path.join(dataset_path, "train", "images")
labels_path = os.path.join(dataset_path, "train", "labels")

# Get all images
image_files = os.listdir(images_path)

# Select 10 random images
random_images = random.sample(image_files, 10)

# Create figure
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for ax, image_file in zip(axes, random_images):

    # Read image
    image_path = os.path.join(images_path, image_file)
    image = cv2.imread(image_path)

    # Convert BGR → RGB
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Image dimensions
    h, w, _ = image.shape

    # Corresponding label file
    label_file = os.path.splitext(image_file)[0] + ".txt"
    label_path = os.path.join(labels_path, label_file)

    # Read labels
    if os.path.exists(label_path):

        with open(label_path, "r") as f:
            labels = f.readlines()

        for label in labels:

            parts = label.strip().split()

            if len(parts) != 5:
                continue

            class_id, x_center, y_center, box_width, box_height = map(float, parts)

            # YOLO → pixel coordinates
            x_center *= w
            y_center *= h
            box_width *= w
            box_height *= h

            x1 = int(x_center - box_width / 2)
            y1 = int(y_center - box_height / 2)
            x2 = int(x_center + box_width / 2)
            y2 = int(y_center + box_height / 2)

            # Draw bounding box
            cv2.rectangle(
                image,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                2
            )

            # Class name
            cv2.putText(
                image,
                f"Class {int(class_id)}",
                (x1, max(y1 - 5, 15)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0, 255, 0),
                2
            )

    # Show image
    ax.imshow(image)
    ax.set_title(image_file)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
!pip install ultralytics -q

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")

In [ ]:
yaml_path = "/content/Ambulance-Fire-Police-5/data.yaml"

with open(yaml_path, "r") as f:
    content = f.read()

lines = content.splitlines()

new_lines = []

for line in lines:
    if line.startswith("train:"):
        new_lines.append("train: train/images")

    elif line.startswith("val:"):
        new_lines.append("val: val/images")

    elif line.startswith("test:"):
        new_lines.append("test: test/images")

    else:
        new_lines.append(line)

content = "\n".join(new_lines) + "\n"

with open(yaml_path, "w") as f:
    f.write(content)

print("data.yaml updated successfully!\n")
print(content)

In [ ]:
model.train(
    data="/content/Ambulance-Fire-Police-5/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16
)

In [ ]:
from ultralytics import YOLO

model = YOLO("runs/detect/train-8/weights/best.pt")

results = model.predict(
    source="Test.png",
    conf=0.5,
    save=True,
    show=False
)

In [ ]:
from google.colab import files

files.download("runs/detect/train-8/weights/best.pt")